离散向量（Discrete Vector）
2.1 独热编码（One-Hot Encoding）
2.2 词袋模型（Bag-of-Words, BoW）
2.2.2 余弦相似度计算 ->余弦相似度（Cosine Similarity）
2.3 TF-IDF（Term Frequency-Inverse Document Frequency）
正如 Salton 等人所指出的，优秀的索引项应该能降低文档空间的密度，使语义不同的文档在空间中距离更远
TF-IDF 的理念是一个词的重要性与其在当前文档中出现的次数成正比，而与其在整个语料库中出现的频率成反比。换言之，一个词在当前文档里越常见，但在其他文档里越罕见，它的权重就越高。


TF-IDF 主要由两部分组成，第一部分是词频（Term Frequency, TF），用于衡量一个词在当前文档中出现的频繁程度。它的计算方式主要有两种，一是直接使用原始频数,     二是采用归一化频率, 第二部分是逆文档频率（Inverse Document Frequency, IDF），用于衡量一个词的"稀有"程度或"信息量"。这一概念由 Karen Sparck Jones 在 1972 年提出

N-gram（1-gram 是特例）关心的是“词的顺序”。它的核心基于马尔可夫假设，也就是认为一个词出现的概率只取决于它前面 N-1 个词。
根据依赖的前文长度不同，N-gram 模型可分为多种类型，其中 Unigram（1-gram） 和词袋模型一样，假设每个词独立且不依赖前文；Bigram（2-gram） 只依赖前 1 个词，例如看到“喜欢”预测“玩”的概率；而 Trigram（3-gram） 则依赖前 2 个词，例如根据“喜欢 玩”来预测“GTA6”的概率。
现在的 GPT 等大模型，本质上可以被视为一个 N 非常非常大（比如 N=128000）的超级 N-gram 模型。它们都在执行根据上文预测下一个词这一任务。


现代深度学习模型，尤其是大语言模型，采用的是一种更简洁、更灵活的输入方式——序号化（Sequentialization）。在深度学习中，通常只进行最少的预处理。不会再像传统方法那样，费尽心思地设计复杂的特征工程（如计算 TF-IDF）来告诉模型哪些词重要。相反，只需要把文本转换成最基础的整数 ID 序列，然后把"学习词语的含义和重要性"这个更复杂的任务，交给模型自己去完成。

3.1 序号化过程
序号化，也称整数编码，是将分词后的词元序列转换为深度学习模型能够处理的整数序列的核心步骤。其过程如下：

（1）构建词典：与 One-Hot 类似，首先从训练语料中构建一个词典。但在深度学习中，这个词典通常是字级别的（如 BERT），或是子词级别的（如 GPT），而不是词级别的。

（2）增加特殊词元：在词典中加入一些有特殊功能的 Token，至少包括 [PAD]（Padding）和 [UNK]（Unknown）。[PAD] 的 ID 通常为 0，用于将短句子填充至同一批次内的最长长度，以满足批处理需求；[UNK] 的 ID 通常为 1，用于表示所有词典中未出现过的词。根据任务需求，还可能加入 [CLS]（分类）、[SEP]（分隔）等其他特殊词元。

（3）ID 映射：将文本序列中的每个词元（字/子词）直接映射为其在词典中的整数 ID。

在实践中，很少从零开始为自己的小数据集构建词典。更常见的做法是，直接使用像 BERT、GPT 这类预训练模型官方提供的词典文件（vocab.txt）。这些词典通常包含了数万个字、子词、符号等，是在海量通用语料上构建的，覆盖面非常广。例如，Google 的中文 BERT 模型词典 vocab.txt 中就包含了约 21128 个词元，其中不仅有常用汉字，还包括了英文字母、数字、标点及 [PAD]/[UNK] 等特殊符号

分布式表示（Distributed Representation）被提出，目的是将词语映射到一个低维、稠密、且蕴含丰富语义信息的连续向量空间中。理想中的词向量需要同时满足语义蕴含和低维稠密两个主要目标。语义蕴含要求向量之间的距离能够度量词语之间的语义相似度，这背后的原理就是分布式假设的朴素应用，也就是说如果两个词经常在相似的上下文中共同出现，那么它们的向量在空间上应该是彼此靠近的。

二、主题模型
题模型是基于机器学习和传统数学思想的经典方法。它尝试从宏观的视角，通过分析大量文档的词语共现统计，来发现词语间的潜在语义关联。它的关键假设是一篇文档由多个“主题”按一定比例混合而成，而一个主题又由多个“词语”按一定概率组成。词语之所以会一同出现在某篇文档中，是因为它们都在共同描述这篇文章所包含的某个或某些潜在主题。例如，一篇关于"人工智能"的文档，会高频出现"深度学习"、"Transformer"、"注意力机制"等词。正是因为这些词都强关联于“AI技术”这个主题，它们才频繁地共现在一起。所以，一个词的向量，就可以用它与各个主题的关联强度来表示。而这其中最核心的技术就是矩阵分解（Matrix Factorization）

三、Word2Vec
它的思想来源于语言学中的分布式假设，即一个词的含义，由其上下文中的词语所决定 3。换言之，如果两个词的上下文经常是相似的，那么这两个词的语义就是相近的。

Word2Vec 通常被认为是一种浅层神经网络模型（Shallow Neural Network）。
理解 Word2Vec 的关键在于区分它的最终目标和实现手段，神经网络结构本身只是获取词向量的一种方式，并非模型的最终目的。

Word2Vec 的最终目标是获取一个高质量的词向量查询表，本质上是一个巨大的矩阵
Win，其中每一行都是对应单词的稠密向量。为达成此目标，Word2Vec 设计了一个巧妙的“伪任务”作为实现手段，也就是根据上下文预测中心词（或反之），并在此过程中将词向量查询表作为模型参数进行训练和优化。训练结束后，执行预测任务的神经网络本身会被丢弃，我们真正保留和使用的，只有作为其内部参数的那个词向量查询表。

Word2Vec 包含 CBOW 和 Skip-gram 两种具体的实现模型。
CBOW（Continuous Bag-of-Words）的任务是 “根据上下文预测中心词”。
与 CBOW 恰好相反，Skip-gram 的任务是 “根据中心词预测上下文”。

SyntaxError: invalid syntax (1360572425.py, line 1)